# 03 - Desenho do Experimento (Teste A/B)

Objetivo: validar em producao se substituir a regra atual pela regra champion de priorizacao aumenta a chance de contato efetivo com o cidadao.

Estimando principal: efeito da regra champion sobre a proporcao de CPFs elegiveis com pelo menos uma entrega (`delivered` ou `read`) entre os telefones selecionados.

A unidade de randomizacao e de analise e o CPF. A populacao principal do teste sao CPFs com 2 ou mais telefones moveis candidatos, pois sao os casos em que o algoritmo de escolha altera a decisao operacional.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.power import zt_ind_solve_power
from statsmodels.stats.proportion import confint_proportions_2indep, proportion_effectsize, proportions_ztest

import utils as u

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Imports OK')


## 1. Baseline historico em CPFs elegiveis

O baseline do experimento deve representar apenas a populacao impactada pela regra de escolha: CPFs com pelo menos 2 telefones moveis candidatos. CPFs com 1 telefone continuam importantes para cobertura, mas nao entram na analise principal porque nao ha decisao de ranking a validar.

Na base mascarada, o CPF da dimensao de telefone nao casa diretamente com o CPF dos logs de disparo. Por isso, o baseline historico usa uma proxy observavel de elegibilidade: CPFs que ja tiveram 2 ou mais telefones distintos nos disparos. A dimensao ainda e carregada para medir o tamanho potencial da populacao elegivel no RMI.


In [ ]:
df_disparo, df_telefone = u.carregar_dados()
df_disparo = u.filtrar_status_invalidos(df_disparo)
df_telefone = u.filtrar_telefones_fixos(df_telefone)
df_disparo['envio_datahora'] = pd.to_datetime(df_disparo['envio_datahora'])

df_aparicoes_brutas = u.explodir_aparicoes(df_telefone)
df_phone_cpf_dim = u.preparar_telefone_cpf(df_aparicoes_brutas)
df_phone_cpf_dim['cpf_key_dim'] = df_phone_cpf_dim['cpf'].astype('string')

qtd_telefones_cpf_dim = df_phone_cpf_dim.groupby('cpf_key_dim')['telefone_numero'].nunique()
cpfs_elegiveis_dimensao = qtd_telefones_cpf_dim[qtd_telefones_cpf_dim >= 2].index

df_disparo_cpf = df_disparo.dropna(subset=['cpf']).copy()
df_disparo_cpf['cpf_key'] = df_disparo_cpf['cpf'].astype('string')
qtd_telefones_historico_cpf = df_disparo_cpf.groupby('cpf_key')['contato_telefone'].nunique()
cpfs_elegiveis_historico = qtd_telefones_historico_cpf[qtd_telefones_historico_cpf >= 2].index

metricas_cpf = u.preparar_metricas_cpf(df_disparo)
metricas_cpf['cpf_key'] = metricas_cpf['cpf'].astype('string')
metricas_cpf_elegiveis = metricas_cpf[metricas_cpf['cpf_key'].isin(cpfs_elegiveis_historico)].copy()

if metricas_cpf_elegiveis.empty:
    raise ValueError('Nao ha CPFs elegiveis com disparos historicos para estimar o baseline do experimento.')

taxa_entrega_cpf_baseline_elegivel = metricas_cpf_elegiveis['cpf_teve_entrega'].mean()
taxa_read_cpf_baseline_elegivel = metricas_cpf_elegiveis['cpf_teve_read'].mean()
taxa_falha_cpf_baseline_elegivel = metricas_cpf_elegiveis['cpf_teve_falha'].mean()

dias = max((df_disparo['envio_datahora'].max() - df_disparo['envio_datahora'].min()).days, 1)
cpfs_historicos_com_disparo = metricas_cpf['cpf_key'].nunique()
cpfs_elegiveis_com_disparo = metricas_cpf_elegiveis['cpf_key'].nunique()
cpfs_elegiveis_por_dia = cpfs_elegiveis_com_disparo / dias
cobertura_elegivel_historica = cpfs_elegiveis_com_disparo / cpfs_historicos_com_disparo

caminho_resumo_priorizacao = u.PROCESSED_DIR / 'resumo_modelo_priorizacao.pkl'
if caminho_resumo_priorizacao.exists():
    with open(caminho_resumo_priorizacao, 'rb') as f:
        resumo_modelo_priorizacao = pickle.load(f)
else:
    resumo_modelo_priorizacao = {}

CHAMPION_METODO = resumo_modelo_priorizacao.get('champion_metodo', 'champion_notebook_02')
CHAMPION_MOTIVO = resumo_modelo_priorizacao.get('champion_motivo', 'Executar o notebook 02 para materializar o champion escolhido.')

display(pd.DataFrame([{
    'cpfs_historicos_com_disparo': cpfs_historicos_com_disparo,
    'cpfs_elegiveis_dimensao_rmi': len(cpfs_elegiveis_dimensao),
    'cpfs_elegiveis_com_disparo': cpfs_elegiveis_com_disparo,
    'cobertura_elegivel_historica': cobertura_elegivel_historica,
    'criterio_elegibilidade_baseline': '2+ telefones distintos observados nos logs por CPF',
    'taxa_entrega_cpf_baseline_elegivel': taxa_entrega_cpf_baseline_elegivel,
    'taxa_read_cpf_baseline_elegivel': taxa_read_cpf_baseline_elegivel,
    'taxa_falha_cpf_baseline_elegivel': taxa_falha_cpf_baseline_elegivel,
    'periodo_dias': dias,
    'cpfs_elegiveis_por_dia_estimado': cpfs_elegiveis_por_dia,
    'champion_metodo': CHAMPION_METODO,
}]))

print('Motivo do champion:', CHAMPION_MOTIVO)


## 2. Hipoteses e metricas

| Item | Definicao |
|---|---|
| H0 | A regra champion nao aumenta a proporcao de CPFs elegiveis com pelo menos uma entrega. |
| H1 | A regra champion aumenta a proporcao de CPFs elegiveis com pelo menos uma entrega. |
| Metrica primaria | CPF elegivel teve pelo menos uma entrega (`delivered` ou `read`). |
| Secundarias | CPF teve pelo menos um `read`, taxa de falha por CPF, custo por entrega e cobertura elegivel. |
| Guardrails | Bloqueios/spam, incidentes de API, desequilibrio por `categoria_hsm`, DDD e quantidade de telefones. |


## 3. Mecanica do teste

- Randomizacao por CPF, persistente por hash, nunca por telefone.
- Split 50/50 entre controle e tratamento.
- Estratificar por `categoria_hsm`, DDD e faixa de quantidade de telefones candidatos quando operacionalmente possivel.
- Controle: regra atual documentada; se a regra atual nao estiver formalizada, usar baseline deterministico alfabetico/ordem de cadastro descrito antes do teste.
- Tratamento: `CHAMPION_METODO` produzido pelo notebook 02.
- Ambos os grupos escolhem 2 telefones por CPF elegivel, preservando comparabilidade de custo.
- Excluir CPFs com menos de 2 telefones candidatos da analise principal; reporta-los em analise separada de cobertura.


## 4. Tamanho amostral


In [ ]:
p1 = taxa_entrega_cpf_baseline_elegivel
uplifts = [0.005, 0.01, 0.02, 0.03]
alpha = 0.05
power = 0.80

resultados_tamanho = []
for uplift in uplifts:
    p2 = min(p1 + uplift, 0.99)
    effect_size = proportion_effectsize(p2, p1)
    n_por_grupo = zt_ind_solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1, alternative='larger')
    n_total = int(np.ceil(n_por_grupo * 2))
    resultados_tamanho.append({
        'uplift_pp': uplift * 100,
        'p1_entrega_cpf_elegivel': p1,
        'p2_tratamento': p2,
        'n_por_grupo': int(np.ceil(n_por_grupo)),
        'n_total_cpfs': n_total,
        'dias_necessarios': int(np.ceil(n_total / cpfs_elegiveis_por_dia)) if cpfs_elegiveis_por_dia > 0 else None,
    })

df_tamanho = pd.DataFrame(resultados_tamanho)
display(df_tamanho)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_tamanho['uplift_pp'], df_tamanho['n_total_cpfs'], marker='o', color='steelblue', linewidth=2)
for _, row in df_tamanho.iterrows():
    ax.annotate(f"{int(row['n_total_cpfs']):,}", (row['uplift_pp'], row['n_total_cpfs']), textcoords='offset points', xytext=(0, 10), ha='center')
ax.set_xlabel('Uplift minimo detectavel (p.p.)')
ax.set_ylabel('Tamanho amostral total (CPFs)')
ax.set_title('Tamanho amostral para entrega em CPFs elegiveis')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Duracao estimada e analise


In [ ]:
print('Duracao estimada do teste A/B')
for _, row in df_tamanho.iterrows():
    dias_teste = row['dias_necessarios']
    semanas = dias_teste / 7 if dias_teste else None
    print(f"Uplift de {row['uplift_pp']:.1f} p.p.: {int(row['n_total_cpfs']):,} CPFs elegiveis -> ~{dias_teste} dias ({semanas:.1f} semanas)")

print('\nRecomendacao: rodar no minimo 14 dias para capturar efeitos de dia da semana, mesmo se o tamanho amostral for atingido antes.')


In [ ]:
print('Codigo de analise apos o teste:')
print('entrega_controle = ...')
print('total_controle = ...')
print('entrega_tratamento = ...')
print('total_tratamento = ...')
print('successes = [entrega_controle, entrega_tratamento]')
print('samples = [total_controle, total_tratamento]')
print('z_stat, p_value = proportions_ztest(successes, samples, alternative="smaller")')
print('ci_low, ci_high = confint_proportions_2indep(entrega_tratamento, total_tratamento, entrega_controle, total_controle, method="wald")')

entrega_controle = 4200
total_controle = 5000
entrega_tratamento = 4300
total_tratamento = 5000
successes = [entrega_controle, entrega_tratamento]
samples = [total_controle, total_tratamento]
z_stat, p_value = proportions_ztest(successes, samples, alternative='smaller')
ci_low, ci_high = confint_proportions_2indep(
    entrega_tratamento, total_tratamento,
    entrega_controle, total_controle,
    method='wald',
)

taxa_controle = entrega_controle / total_controle
taxa_tratamento = entrega_tratamento / total_tratamento
diferenca_absoluta = taxa_tratamento - taxa_controle
lift_relativo = diferenca_absoluta / taxa_controle

print('\nDemonstracao hipotetica')
print('Controle:', f'{entrega_controle}/{total_controle} = {taxa_controle:.2%}')
print('Tratamento:', f'{entrega_tratamento}/{total_tratamento} = {taxa_tratamento:.2%}')
print('Diferenca absoluta:', f'{diferenca_absoluta:.2%}')
print('Lift relativo:', f'{lift_relativo:.2%}')
print('IC 95% da diferenca tratamento-controle:', f'[{ci_low:.2%}, {ci_high:.2%}]')
print('Z-statistic:', f'{z_stat:.4f}')
print('P-value:', f'{p_value:.4f}')


## 6. Plano de analise e criterios de decisao

- Analise principal por intention-to-treat: cada CPF conta no grupo sorteado, mesmo se algum envio falhar operacionalmente.
- Teste estatistico: z-test unilateral para duas proporcoes, avaliando se tratamento > controle na metrica primaria de entrega por CPF elegivel.
- Reportar diferenca absoluta, intervalo de confianca de 95%, lift relativo, leitura por CPF, falha por CPF e custo por entrega.
- Analises de sensibilidade: remover janelas com incidente de API e quebrar resultado por `categoria_hsm`, DDD e faixa de quantidade de telefones.
- Criterio ship: ganho estatisticamente significativo na metrica primaria, sem piora relevante nos guardrails.
- Criterio iterar: ganho positivo mas inconclusivo, ou ganho concentrado em subgrupos que exija nova regra.
- Criterio nao substituir: ausencia de ganho, piora operacional relevante ou risco elevado de contato com destinatario errado.

## 7. Riscos e mitigacoes

| Risco | Mitigacao |
|---|---|
| Contaminacao por telefone compartilhado | Randomizar por CPF e monitorar overlap de telefones entre grupos. |
| Atraso de status (`delivered`/`read`) | Fechar a janela de mensuracao somente apos prazo operacional suficiente para estabilizar status. |
| Sazonalidade e mix de campanhas | Rodar pelo menos duas semanas, estratificar por `categoria_hsm` e controlar por dia da semana. |
| Controle mal definido | Fixar e documentar a regra controle antes do teste. |
| CPFs com menos de 2 telefones | Excluir da analise principal e reportar cobertura. |
| Incidentes de API ou fornecedor | Marcar janelas afetadas e rodar analise de sensibilidade sem esses periodos. |
| Custo desigual entre grupos | Garantir que controle e tratamento selecionem 2 telefones por CPF elegivel. |
| Destinatario errado | Monitorar telefones com muitos proprietarios e pioras por esse segmento como guardrail. |


## 8. Artefato do experimento


In [ ]:
u.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
parametros_experimento = {
    'baseline_entrega_cpf_elegivel': taxa_entrega_cpf_baseline_elegivel,
    'baseline_read_cpf_elegivel': taxa_read_cpf_baseline_elegivel,
    'baseline_falha_cpf_elegivel': taxa_falha_cpf_baseline_elegivel,
    'cpfs_historicos_com_disparo': int(cpfs_historicos_com_disparo),
    'cpfs_elegiveis_dimensao_rmi': int(len(cpfs_elegiveis_dimensao)),
    'cpfs_elegiveis_historico_com_disparo': int(cpfs_elegiveis_com_disparo),
    'criterio_elegibilidade_baseline': '2+ telefones distintos observados nos logs por CPF',
    'cobertura_elegivel_historica': float(cobertura_elegivel_historica),
    'tamanhos_amostrais': df_tamanho.to_dict('records'),
    'alpha': alpha,
    'power': power,
    'alternative': 'tratamento_maior_que_controle',
    'unidade_randomizacao': 'cpf',
    'populacao_principal': 'cpfs_com_2_ou_mais_telefones_moveis_candidatos',
    'metrica_primaria': 'cpf_teve_ao_menos_uma_entrega',
    'metricas_secundarias': ['cpf_teve_ao_menos_um_read', 'taxa_falha_por_cpf', 'custo_por_entrega', 'cobertura_elegivel'],
    'tratamento': CHAMPION_METODO,
    'tratamento_motivo': CHAMPION_MOTIVO,
    'controle': 'regra_atual_documentada_ou_baseline_deterministico_alfabetico_ordem_cadastro',
    'volume_diario_cpfs_elegiveis': cpfs_elegiveis_por_dia,
}
with open(u.PROCESSED_DIR / 'parametros_experimento.pkl', 'wb') as f:
    pickle.dump(parametros_experimento, f)
print('Parametros do experimento salvos em:', u.PROCESSED_DIR / 'parametros_experimento.pkl')
